## Basic trying things (skip)

In [ ]:
import kdsource.api as kds
import numpy as np

In [ ]:
help(kds.write_mcpl)

In [ ]:
file_path = "my_data.mcpl.gz"

empty_10 = np.zeros(10)
full_10 = np.ones(10)
kds.write_mcpl(filename=file_path, ekin=full_10, x=empty_10, y=empty_10, z=empty_10, ux=full_10, uy=empty_10, uz=empty_10, time=empty_10, weight=full_10, pdgcode=2112)

In [ ]:
import mcpl
import pandas as pd

myfile=mcpl.MCPLFile(file_path,blocklength=2)
datas = []
for i,p in enumerate(myfile.particle_blocks):
    print("block no.", i)
    data = pd.DataFrame({
        'Kf': p.pdgcode,  # Particle PDG code
        'x': p.x,  # x-position
        'y': p.y,  # y-position
        'z': p.z,  # z-position
        'Vx': p.ux,  # Velocity in x-direction
        'Vy': p.uy,  # Velocity in y-direction
        'Vz': p.uz,  # Velocity in z-direction
        'e': p.ekin,  # Kinetic energy
        'Wt': p.weight  # Weight of the particle
    })
    datas.append(data)
data = pd.concat(datas, ignore_index=True)
data

In [ ]:
data.dtypes

In [ ]:
myfile=mcpl.MCPLFile("./examples/uniform/output.mcpl.gz",blocklength=100)
datas = []
for i,p in enumerate(myfile.particle_blocks):
    if i==1: break
    data = pd.DataFrame({
        'Kf': p.pdgcode,  # Particle PDG code
        'x': p.x,  # x-position
        'y': p.y,  # y-position
        'z': p.z,  # z-position
        'Vx': p.ux,  # Velocity in x-direction
        'Vy': p.uy,  # Velocity in y-direction
        'Vz': p.uz,  # Velocity in z-direction
        'e': p.ekin,  # Kinetic energy
        'Wt': p.weight  # Weight of the particle
    })
    datas.append(data)
data = pd.concat(datas, ignore_index=True)
data

In [ ]:
kds.write_mcpl(filename=file_path, ekin=data["e"], x=data["x"], y=data["y"], z=data["z"],
                ux=data["Vx"], uy=data["Vy"], uz=data["Vz"], time=np.zeros_like(data["Kf"]), weight=data["Wt"], pdgcode=data["Kf"])

In [ ]:
import kdsource.api as kds  # Importing the KDSource Python API

In [ ]:
kds.Geometry?

In [ ]:
kds.Metric?

In [ ]:
kds.geom._metrics

In [ ]:
my_energy = kds.geom.Energy()
#my_lethargy = kds.geom.Lethargy()

In [ ]:
type(my_energy)

In [ ]:
for k,v in my_energy.__dict__.items():
    print(k, str(v))

In [ ]:
# for k,v in my_lethargy.__dict__.items():
#     print(k, str(v))

In [ ]:
my_position = kds.geom.Vol()
my_time = kds.geom.Time()
my_directions = kds.geom.Isotrop()

In [ ]:
my_geometry = kds.Geometry([
    my_energy, my_time, my_position, my_directions
])

In [ ]:
for k,v in my_geometry.__dict__.items():
    print(k, str(v))

In [ ]:
kds.KDSource?

In [ ]:
import kdsource

In [ ]:
kdsource?

In [ ]:
kds.kde.bw_methods

In [ ]:
# Step 1: Load your original MCPL particle track list file
my_input_file_path = "./examples/uniform/output.mcpl.gz"

my_input_plist = kds.PList(filename=my_input_file_path,
                           readformat="mcpl",
                           pt="n",
                           trasl=None,
                           rot=None,
                           switch_x2z=False,
                           set_params=True)


In [ ]:
vars(my_input_plist)

In [ ]:
my_source.fitted

In [ ]:
my_source = kds.KDSource(
    plist=my_input_plist, geom=my_geometry, bw="mlcv", J=1.0, kernel="gaussian"
)

In [ ]:
my_source.fitted

In [ ]:
[attr for attr in dir(my_source) if attr[0:2]!="__"]

In [ ]:
vars(my_source.plist)

In [ ]:
my_source.fitted, my_source.bw_method

In [ ]:
my_source.fit(N=10000)

In [ ]:
vars(my_source.geom)

In [ ]:
my_source.kde.fit?

In [ ]:
my_source.fitted

In [ ]:
import numpy as np
my_source.plot_t(np.array([-1,0,1]))

In [ ]:
my_source.kde.data

In [ ]:
my_source.plot_E(grid_E=np.array([0,1e-8,2e-8,3e-8]))

In [ ]:
my_source.plot_integr(var="x", grid=np.arange(-5,5,0.2));

In [ ]:
my_source.save()

In [ ]:
kds.resample_to_mcpl("_source.xml", "generated_from_source.mcpl.gz", nout=100)

In [ ]:
import mcpl
import pandas as pd

gen_file=mcpl.MCPLFile("generated_from_source.mcpl.gz")
datas = []
for i,p in enumerate(gen_file.particle_blocks):
    print("block no.", i)
    data = pd.DataFrame({
        'Kf': p.pdgcode,  # Particle PDG code
        'x': p.x,  # x-position
        'y': p.y,  # y-position
        'z': p.z,  # z-position
        'Vx': p.ux,  # Velocity in x-direction
        'Vy': p.uy,  # Velocity in y-direction
        'Vz': p.uz,  # Velocity in z-direction
        'e': p.ekin,  # Kinetic energy
        'Wt': p.weight  # Weight of the particle
    })
    datas.append(data)
data = pd.concat(datas, ignore_index=True)
data

In [ ]:
my_input_plist.N

In [ ]:
kdsource.plist.pt2pdg?

## methods
read_data, resample_df, resample_df_gen, generate_data, plot_data_row

## method definitions (please run this)

In [1]:
import kdsource.api as kds
import mcpl
import pandas as pd
from pathlib import Path
import tempfile
import numpy as np


def is_constant_column(column):
    return np.all(column == column[0], axis=0)


def read_data(file_path, n=-1):
    """
    Read up to n particles from the mcpl file. n==-1 means read all.
    """
    blocklength = 10000
    gen_file = mcpl.MCPLFile(file_path, blocklength=blocklength)
    datas = []
    for i, p in enumerate(gen_file.particle_blocks):
        data = pd.DataFrame(
            {
                "ekin": p.ekin,  # Kinetic energy
                "x": p.x,  # x-position
                "y": p.y,  # y-position
                "z": p.z,  # z-position
                "ux": p.ux,  # Velocity in x-direction
                "uy": p.uy,  # Velocity in y-direction
                "uz": p.uz,  # Velocity in z-direction
                "time": p.time,
                "weight": p.weight,  # Weight of the particle
                "pdgcode": p.pdgcode,  # Particle PDG code
            }
        )
        datas.append(data)
        if n != -1 and (i + 1) * blocklength >= n:
            break
    data = pd.concat(datas, ignore_index=True)
    if n != -1 and len(data) > n:
        data = data[:n]
    return data

def resample_df(
    source,
    dest,
    nout,
    rng=None,
    force=False,
    no_df=False,
    del_dest=False,
    del_source=False,
):
    """
    Generates data at `dest` and by default returns it as a pandas dataframe.
    - source: source model .xml file (which contains the original data path that is required for sampling)
    - dest: a file path with .mcpl.gz ending
    - nout: amount of particles generated
    - rng: rng parameter
    - force: whether `dest` file may be overwritten if it already exists
    - no_df: whether you do NOT want to receive a dataframe for some reason
    - del_dest: whether to delete file at `dest` in the end
    - del_source: whether or not to delete the source .xml file in the end

    Returns
    - pandas dataframe, or `dest`, or None
    """
    # print("Amount of particles to generate:", str(nout))
    kds.resample_to_mcpl(
        kds_sourcefile=source,
        destination_mcpl=dest,
        nout=nout,
        rng=rng,
        force=force,
    )
    # print("Data generated at", dest)
    if not no_df:
        data = read_data(dest)
    if del_dest:
        Path(dest).unlink()
    if del_source:
        Path(source).unlink()
    if not no_df:
        return data
    if not del_dest:
        return dest
    return None

def resample_df_gen(source, dest, nout, rng=None, del_dest=False, del_source=False):
    """
    A generator for dataframes via the resample method using an intermediate mcpl.gz file at `dest`. ATTENTION: Overwrites `dest` in each iteration!
    - source: source model .xml file (which contains the original data path that is required for sampling)
    - dest: a file path with .mcpl.gz ending
    - nout: amount of particles per dataframe
    - rng: If you specify it, it will be used for all the generated data. If None, uses new random rng for each iteration.
    - del_dest: whether or not to delete the most recent intermediate mcpl.gz file on the disk
    - del_source: whether or not to delete the source .xml file in the end (after the generator has stopped)
    """
    try:
        while True:
            yield resample_df(
                source=source,
                dest=dest,
                nout=nout,
                rng=rng,
                force=True,
                no_df=False,
                del_dest=del_dest,
                del_source=False,
            )
    finally:
        if del_source:
            Path(source).unlink()




def generate_data(
    inputFile,
    output_path=None,
    overwrite_output=False,
    n_gen=1_000,
    pt="n",
    n_fit=-1,
    kernel="gaussian",
    bw="silv",
    pCurrent=1.0,
    model_path=None,
    overwrite_model=False,
    generator=False,
):
    """
    Generates data similar to inputFile using kdsource, which uses KDE (Kernel Density Estimation) modeling.

    Parameters
    ---
    - inputFile: file path to input data; requires ending `.mcpl` or `.mcpl.gz`
    - output_path: file path for generated data (optional, see "Returns" section); file ending will be `.mcpl.gz`
    - overwrite_output: whether already existing file at output_path may be overwritten
    - n_gen: amount of particles to generate
    - pt: particle type ("n", "p", or "e") of generated particles and of input particles used for fitting the KDE model
    - n_fit: amount of input particles (with type `pt`) used for fitting of the source model (-1 means all)
    - kernel: kernel shape ("gaussian", "box", or "epa" (Epanechnikov))
    - bw: method for finding kernel bandwidth ("silv", "knn" or "mlcv")
    - pCurrent: total source particle current
    - model_path: where to save the model xml file (None makes it temporary)
    - overwrite_model: whether already existing file at model_path may be overwritten
    - generator: whether you want a generator for dataframes instead of one dataframe. `n_gen` will be the amount per yield.

    Returns
    ---
    - If output_path is specified: None. (The data will be saved to the file.)
    - If output_path is None: Pandas dataframe. Or, if `generator` is True: Generator of pandas dataframes.
    """

    inputFile = str(Path(inputFile).absolute())

    return_a_df = False
    output_is_temp = False
    model_is_temp = False

    if output_path is not None:
        output_path = Path(output_path).absolute().with_suffix(".mcpl.gz")
        if output_path.exists() and not overwrite_output:
            raise ValueError(
                "Output file already exists and overwriting is not allowed"
            )
        output_path = str(output_path)
    else:
        return_a_df = True
        output_path = str(Path(tempfile.gettempdir()) / "brightify_kdsource.mcpl.gz")
        output_is_temp = True

    if model_path is not None:
        model_path = Path(model_path).absolute().with_suffix(".xml")
        if model_path.exists() and not overwrite_model:
            raise ValueError("Model file already exists and overwriting is not allowed")
        model_path = str(model_path)
    else:
        model_path = str(Path(tempfile.gettempdir()) / "brightify_kdsource.xml")
        model_is_temp = True

    my_input_plist = kds.PList(
        filename=inputFile,
        readformat="mcpl",
        pt=pt,
        trasl=None,
        rot=None,
        switch_x2z=False,
        set_params=True,
    )
    # print(vars(my_input_plist))

    my_energy = kds.geom.Energy()
    my_time = kds.geom.Time()
    my_position = kds.geom.Vol()
    my_directions = kds.geom.Isotrop()
    my_geometry = kds.Geometry([my_energy, my_time, my_position, my_directions])
    # print(vars(my_geometry))

    my_source = kds.KDSource(
        plist=my_input_plist, geom=my_geometry, bw=bw, J=pCurrent, kernel=kernel
    )
    # print(vars(my_source))

    if n_fit == -1 or n_fit > my_input_plist.N:
        n_fit = my_input_plist.N
    # print("Amount of particles for fitting:", n_fit)

    my_source.fit(N=n_fit)

    model_path = str(Path(my_source.save(xmlfilename=model_path)).absolute())
    # if not model_is_temp:
    #     print("Source model saved to", model_path)
    # else:
    #     print("Temporary source model saved to", model_path)



    if not return_a_df:
        kds.resample_to_mcpl(
            kds_sourcefile=model_path,
            destination_mcpl=output_path,
            nout=n_gen,
            rng=None,
            force=True,
        )
        if model_is_temp:
            Path(model_path).unlink()
        return output_path
    else:
        if not generator:
            return resample_df(
                source=model_path,
                dest=output_path,
                nout=n_gen,
                rng=None,
                force=True,
                no_df=False,
                del_dest=output_is_temp,
                del_source=model_is_temp,
            )
        else:
            return resample_df_gen(
                source=model_path,
                dest=output_path,
                nout=n_gen,
                rng=None,
                del_dest=output_is_temp,
                del_source=model_is_temp,
            )

In [2]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

def plot_data_row(datas, rows=["x", "ux", "ekin", "weight"]):
    colors = list(mcolors.TABLEAU_COLORS.values())
    for j, row in enumerate(rows):
        color = colors[j]
        for i, data in enumerate(datas):
            plt.hist(data[row], density=True, bins=20, histtype="step", label=row+" "+str(i), color=color, linestyle=(0,(i%2+1,i*2+1)))
    plt.legend()
    plt.show()

## examples (skip)

In [ ]:
example_path = "./examples/uniform/output.mcpl.gz"

In [ ]:
data1 = generate_data(inputFile=example_path, n_gen=100_000, n_fit=1_000_000, bw="silv")

In [ ]:
data2 = read_data(example_path, 100_000)

In [ ]:
gen = generate_data(
    inputFile=example_path,
    output_path=None,
    overwrite_output=False,
    n_gen=100,
    pt="n",
    n_fit=100_000,
    kernel="gaussian",
    bw="silv",
    pCurrent=1.0,
    model_path=None,
    overwrite_model=False,
    generator=True,
)

In [ ]:
data3=next(gen)

In [ ]:
gen.close()

In [ ]:
data3=next(gen)

In [ ]:
import numpy as np

In [ ]:
for d in [data1, data2, data3]:
    d["V_norm"] = np.sqrt((d.Vx*d.Vx + d.Vy*d.Vy + d.Vz+d.Vz))

In [ ]:
data1.describe()

In [ ]:
data2.describe()

In [ ]:
data3.describe()

In [ ]:
plot_data_row(datas=[data1, data2, data3], rows=["e", "x", "Vx"])

# Exercise 26.10. - 30.10.

## file paths and manual inspection

In [3]:
example_path_1 = "./examples/Brightify Fig 6/Fig6a/output.mcpl.gz"  # const: ekin=2.5e-08, weight=1; almost const: z=0.06; n = 20_139_128
example_path_2 = "./examples/Brightify Fig 6/Fig6b/output.mcpl.gz"  # const: ekin=2.5e-08, weight=1; almost const: z=0.06; n = 21_580_959 (SAME AS 5)
example_path_3 = "./examples/Brightify Fig 6/Fig6c/output.mcpl.gz"  # const: ekin=2.5e-08, weight=1; almost const: z=0.06; n = 11_002_813
example_path_4 = "./examples/mcpl_different_normal/dump_final.mcpl.gz"  # almost const: x=13.718; n = 2_738_810
example_path_5 = "./examples/uniform/output.mcpl.gz"  # const: ekin=2.5e-08, weight=1; almost const: z=0.06; n = 21_580_959 (SAME AS 2)
example_path_6 = "./examples/resampled.mcpl.gz"  # const: z=286.915, weight=1; n = 2_000_000

In [ ]:
# !pymcpltool --stats "$example_path_4"

In [ ]:
# !pymcpltool --stats --gui "$example_path_6" # it did not work inside the notebook here, but from terminal (because of WSL)

## compare silverman & mlcv and gaussian & epa, for 2 different data

### declare files, plist parameters and geometries

In [4]:
example_paths = {
    1: example_path_1,
    2: example_path_2,
    3: example_path_3,
    4: example_path_4,
    5: example_path_5, # is the same as 2
    6: example_path_6,
}

# plist parameters:
pt = {index:"n" for index in example_paths.keys()}
trasl = {}
rot = {}
switch_x2z = {4: True} # swaps x to z, y to x, z to y

geometries = { # one per file
    1: kds.geom.Geometry([kds.geom.SurfXY(z=0.06)]), # omitting all dimensions except x and y position
    2: kds.geom.Geometry([kds.geom.SurfXY(z=0.06)]),
    3: kds.geom.Geometry([kds.geom.SurfXY(z=0.06)]),
    4: kds.geom.GeomFlat(z=13.718, E0=8.18e-08),
    5: kds.geom.Geometry([kds.geom.SurfXY(z=0.06)]),
    6: kds.geom.GeomFlat(z=286.915, E0=10),
}

### choose what to calculate

In [26]:
bw_kernel_combinations = {  # choose from these by referencing the number/index below
    1: ("silv", "gaussian"),
    2: ("silv", "epa"),
    3: ("mlcv", "gaussian"),
    4: ("mlcv", "epa"),
}

calculate_these = [  # format: (file_index, bw_kernel_index, n_fit)
    # where n_fit is the number of particles (after "pt" filtering) to use for the fit.
    # type -1 for n_fit to use all the particles. use a value for n_fit between 0 and 1 to use a certain percentage
    # (5, 1, 10_000),
    # (5, 2, 10_000),
    # (5, 3, 10_000),
    # (5, 4, 10_000),
    # (6, 1, 10_000),
    # (6, 2, 10_000),
    # (6, 3, 10_000),
    # (6, 4, 10_000),
]
# calculate_these.extend(
#     [
#         (fn, bw_kernel_index, n_fit)
#         for fn in (5,6)
#         for bw_kernel_index in (3,4)
#         for n_fit in (
#             1_000,
#             5_000,
#             10_000,
#             20_000,
#             50_000,
#             100_000,
#             200_000,
#         )
#     ]
# )

calculate_these.extend(
    [
        (fn, bw_kernel_index, n_fit)
        for fn in (5,6)
        for bw_kernel_index in (1,2)
        for n_fit in (
            1_000,
            5_000,
            10_000,
            20_000,
            50_000,
            100_000,
            200_000,
            500_000,
            1_000_000,
            2_000_000,
        )
    ]
)

calculate_these

[(5, 1, 1000),
 (5, 1, 5000),
 (5, 1, 10000),
 (5, 1, 20000),
 (5, 1, 50000),
 (5, 1, 100000),
 (5, 1, 200000),
 (5, 1, 500000),
 (5, 1, 1000000),
 (5, 1, 2000000),
 (5, 2, 1000),
 (5, 2, 5000),
 (5, 2, 10000),
 (5, 2, 20000),
 (5, 2, 50000),
 (5, 2, 100000),
 (5, 2, 200000),
 (5, 2, 500000),
 (5, 2, 1000000),
 (5, 2, 2000000),
 (6, 1, 1000),
 (6, 1, 5000),
 (6, 1, 10000),
 (6, 1, 20000),
 (6, 1, 50000),
 (6, 1, 100000),
 (6, 1, 200000),
 (6, 1, 500000),
 (6, 1, 1000000),
 (6, 1, 2000000),
 (6, 2, 1000),
 (6, 2, 5000),
 (6, 2, 10000),
 (6, 2, 20000),
 (6, 2, 50000),
 (6, 2, 100000),
 (6, 2, 200000),
 (6, 2, 500000),
 (6, 2, 1000000),
 (6, 2, 2000000)]

### Create partial data files to use for fitting

In [27]:
copy_data_folder = "./copied_data"

In [28]:
from pathlib import Path
copy_data_folder = Path(copy_data_folder).absolute()
if not copy_data_folder.exists():
    Path.mkdir(copy_data_folder)

In [29]:
import numpy as np

copy_files = {(int(fn), int(n_fit)): str(p) for fn, n_fit, p in [str(p.with_suffix("").stem).split("_")+[p] for p in copy_data_folder.glob("*.mcpl.gz")]}
print(copy_files)

for (fn, bw_kernel_num, n_fit) in calculate_these:
    file = example_paths[fn]
    copy_file = str(copy_data_folder / Path(str(fn)+"_"+str(n_fit)+".mcpl.gz"))
    fn_n_fit = (fn, n_fit)
    if fn_n_fit in copy_files.keys():
        continue
    copy_data = read_data(file_path=file, n=n_fit)
    kds.write_mcpl(copy_file,
                    ekin=copy_data["ekin"],
                    x=copy_data["x"],
                    y=copy_data["y"],
                    z=copy_data["z"],
                    ux=copy_data["ux"],
                    uy=copy_data["uy"],
                    uz=copy_data["uz"],
                    time=copy_data["time"],
                    weight=copy_data["weight"][0] if is_constant_column(copy_data["weight"]) else copy_data["weight"],
                    pdgcode=copy_data["pdgcode"][0] if is_constant_column(copy_data["pdgcode"]) else copy_data["pdgcode"],
                    polx=None,
                    poly=None,
                    polz=None,
                    userflags=None,
                    )
    copy_files[fn_n_fit] = copy_file


{(5, 5000): '/home/jhonal/Brightify/copied_data/5_5000.mcpl.gz', (6, 200000): '/home/jhonal/Brightify/copied_data/6_200000.mcpl.gz', (5, 50000): '/home/jhonal/Brightify/copied_data/5_50000.mcpl.gz', (6, 50000): '/home/jhonal/Brightify/copied_data/6_50000.mcpl.gz', (6, 5000): '/home/jhonal/Brightify/copied_data/6_5000.mcpl.gz', (6, 20000): '/home/jhonal/Brightify/copied_data/6_20000.mcpl.gz', (5, 100000): '/home/jhonal/Brightify/copied_data/5_100000.mcpl.gz', (5, 200000): '/home/jhonal/Brightify/copied_data/5_200000.mcpl.gz', (5, 10000): '/home/jhonal/Brightify/copied_data/5_10000.mcpl.gz', (5, 20000): '/home/jhonal/Brightify/copied_data/5_20000.mcpl.gz', (6, 100000): '/home/jhonal/Brightify/copied_data/6_100000.mcpl.gz', (6, 10000): '/home/jhonal/Brightify/copied_data/6_10000.mcpl.gz', (5, 1000): '/home/jhonal/Brightify/copied_data/5_1000.mcpl.gz', (6, 1000): '/home/jhonal/Brightify/copied_data/6_1000.mcpl.gz'}
MCPL: Compressing file 5_500000.mcpl
MCPL: Compressed file into 5_500000.mc

### run calculations, to obtain model xml files

In [30]:
models_folder="./calculated_models"

In [31]:
from pathlib import Path
models_folder = Path(models_folder).absolute()
if not models_folder.exists():
    Path.mkdir(models_folder)

In [32]:
import math
import time
import numpy as np

for i, (fn, bw_kernel_num, n_fit) in enumerate(calculate_these):
    
    fn_n_fit = (fn, n_fit)
    plist = kds.PList(
        filename=copy_files[fn_n_fit],
        readformat="mcpl",
        pt=pt[fn] if fn in pt.keys() else "n",
        trasl=trasl[fn] if fn in trasl.keys() else None,
        rot=rot[fn] if fn in rot.keys() else None,
        switch_x2z=switch_x2z[fn] if fn in switch_x2z.keys() else False,
        set_params=True)

    N = plist.N

    bw, kernel = bw_kernel_combinations[bw_kernel_num]
    if n_fit == -1: n_fit = N
    elif 0 < n_fit < 1: n_fit = math.floor(N * n_fit)
    else: n_fit = max(min(N, n_fit), 0)
    if n_fit == 0:
        print("n_fit became zero, calculation will be skipped now.")
        continue

    source = kds.KDSource(
            plist=plist,
            geom=geometries[fn],
            bw=bw,
            kernel=kernel)
    
    fit_success = False
    start = time.perf_counter()
    mlcv_grid_multiplier = 5
    mlcv_grid_start = (-0.1 if n_fit < 100_000 else -0.5) if n_fit < 500_000 else -2.5
    mlcv_grid_stop = (0.1 if n_fit < 100_000 else 0.5) if n_fit < 500_000 else 2.5
    mlcv_grid_num = 8
    mlcv_attempt = 1    
    while fit_success == False and mlcv_attempt <= 3:
        try:
            if bw=="mlcv":
                grid = np.logspace(mlcv_grid_start, mlcv_grid_stop, mlcv_grid_num)
                print("attempting grid logspace start stop", str(mlcv_grid_start), str(mlcv_grid_stop))
                source.fit(N=n_fit, grid=grid)
            else:
                source.fit(N=n_fit)
        except:
            print("Fitting failed")
            if bw=="mlcv":
                mlcv_grid_start *= mlcv_grid_multiplier
                mlcv_grid_stop *= mlcv_grid_multiplier
        else:
            fit_success = True
        finally:
            if bw=="mlcv":
                mlcv_attempt += 1
            else:
                break        
    end = time.perf_counter()
    elapsed_seconds = int(end-start)
    print(f"Elapsed: {elapsed_seconds} s")

    if fit_success:
        xmlfilename = str(models_folder)+"/file_"+str(fn)+"_"+bw+"_"+kernel+"_"+str(N)+"_"+str(n_fit)+"_"+str(elapsed_seconds)+"s.xml"
        source.save(xmlfilename=xmlfilename)


Using existing file /home/jhonal/Brightify/copied_data/5_1000.mcpl.gz
sum_weights = 1000.0
p2 = 1000.0
N = 1000
N_eff = 1000.0
Using 1000 particles for fit.
Calculating bw ... 
Done
Optimal bw (silv) = [[0.09045002 0.0897851 ]]
Elapsed: 0 s
Successfully saved parameters file /home/jhonal/Brightify/calculated_models/file_5_silv_gaussian_1000_1000_0s.xml
Using existing file /home/jhonal/Brightify/copied_data/5_5000.mcpl.gz
sum_weights = 5000.0
p2 = 5000.0
N = 5000
N_eff = 5000.0
Using 5000 particles for fit.
Calculating bw ... 
Done
Optimal bw (silv) = [[0.07042721 0.07101868]]
Elapsed: 0 s
Successfully saved parameters file /home/jhonal/Brightify/calculated_models/file_5_silv_gaussian_5000_5000_0s.xml
Using existing file /home/jhonal/Brightify/copied_data/5_10000.mcpl.gz
sum_weights = 10000.0
p2 = 10000.0
N = 10000
N_eff = 10000.0
Using 10000 particles for fit.
Calculating bw ... 
Done
Optimal bw (silv) = [[0.06203614 0.06328297]]
Elapsed: 0 s
Successfully saved parameters file /home/jh

### generate (resample) data from the models

In [33]:
gen_N = 2_000_000 # number of particles to generate per model
gen_data_folder = "./generated_data"
RNG = 123456789

In [34]:
from pathlib import Path

gen_data_folder = Path(gen_data_folder).absolute()
if not gen_data_folder.exists():
    Path.mkdir(gen_data_folder)

In [38]:
for model_path in models_folder.glob("*"):
    #print(model_path)
    stem = model_path.with_suffix("").stem
    f, fn, bw, kernel, N, n_fit, elapsed = stem.split(sep="_")
    # print(f, fn, bw, kernel, N, n_fit, elapsed)
    fn = int(fn)
    N = int(N)
    n_fit = int(n_fit)
    dest_path = gen_data_folder / Path(stem).with_suffix(".mcpl.gz")
    # print(dest_path)
    if dest_path in gen_data_folder.glob("*"):
        print("skipping", dest_path)
        continue
    resample_df(source=model_path, dest=dest_path, nout=gen_N, rng=RNG, force=True, no_df=True, del_dest=False, del_source=False)

skipping /home/jhonal/Brightify/generated_data/file_5_mlcv_gaussian_100000_100000_1788s.mcpl.gz
Reading xmlfile /home/jhonal/Brightify/calculated_models/file_5_silv_gaussian_10000_10000_0s.xml...
Done.
Resampling...
MCPL: Compressing file file_5_silv_gaussian_10000_10000_0s.mcpl
MCPL: Compressed file into file_5_silv_gaussian_10000_10000_0s.mcpl.gz
Successfully sampled 2000000 particles.
skipping /home/jhonal/Brightify/generated_data/file_5_mlcv_gaussian_5000_5000_7s.mcpl.gz
Reading xmlfile /home/jhonal/Brightify/calculated_models/file_6_silv_epa_500000_500000_0s.xml...
Done.
Resampling...
MCPL: Compressing file file_6_silv_epa_500000_500000_0s.mcpl
MCPL: Compressed file into file_6_silv_epa_500000_500000_0s.mcpl.gz
Successfully sampled 2000000 particles.
skipping /home/jhonal/Brightify/generated_data/file_5_mlcv_epa_10000_10000_15s.mcpl.gz
Reading xmlfile /home/jhonal/Brightify/calculated_models/file_5_silv_gaussian_100000_100000_0s.xml...
Done.
Resampling...
MCPL: Compressing file fi

### read data and produce metrics

In [39]:
from collections import defaultdict
def nested_dict():
    return defaultdict(nested_dict)
def normalize_nested_dict(d):
    if isinstance(d, defaultdict):
        return {k: normalize_nested_dict(v) for k, v in d.items()}
    return d

gen_data_metrics = nested_dict()
for gen_data_path in gen_data_folder.glob("*"):
    stem = gen_data_path.with_suffix("").stem
    f, fn, bw, kernel, N, n_fit, elapsed = stem.split(sep="_")
    # print(elapsed)
    fn = int(fn)
    bw_kernel = (bw, kernel)
    n_fit = int(n_fit)
    elapsed = int(elapsed.rstrip("s"))
    gen_data_metrics[fn][bw_kernel][n_fit]['elapsed'] = elapsed
    gen_data = read_data(gen_data_path)
    summary = gen_data.describe().loc[['mean', 'std']]
    for c in summary.columns:
        gen_data_metrics[fn][bw_kernel][n_fit]['mean'][c] = summary.loc['mean', c]
        gen_data_metrics[fn][bw_kernel][n_fit]['std'][c] = summary.loc['std', c]
gen_data_metrics = normalize_nested_dict(gen_data_metrics)

original_data_metrics = nested_dict()
for fn in gen_data_metrics.keys():
    original_path = example_paths[fn]
    data = read_data(original_path)
    summary = data.describe().loc[['mean', 'std']]
    for c in summary.columns:
        original_data_metrics[fn]['mean'][c] = summary.loc['mean', c]
        original_data_metrics[fn]['std'][c] = summary.loc['std', c]
original_data_metrics = normalize_nested_dict(original_data_metrics)

### save / load

In [40]:
results_dir = "./pickled_results"

In [41]:
import pickle as pkl
from pathlib import Path
results_dir = Path(results_dir).absolute()
    
gen_data_file = str(results_dir / "gen_data.pkl")
orig_data_file = str(results_dir / "orig_data.pkl")
print(gen_data_file, orig_data_file)

/home/jhonal/Brightify/pickled_results/gen_data.pkl /home/jhonal/Brightify/pickled_results/orig_data.pkl


#### save pickle

In [42]:
if not results_dir.exists():
    Path.mkdir(results_dir)
with open(gen_data_file, "wb") as f:
    pkl.dump(gen_data_metrics, f)
with open(orig_data_file, "wb") as f:
    pkl.dump(original_data_metrics, f)

#### load pickle

In [43]:
with open(gen_data_file, "rb") as f:
    gen_data_metrics = pkl.load(f)
with open(orig_data_file, "rb") as f:
    original_data_metrics = pkl.load(f)

In [44]:
gen_data_metrics

{5: {('silv',
   'epa'): {100000: {'elapsed': 0,
    'mean': {'ekin': np.float64(2.5000000292152436e-08),
     'x': np.float64(-8.908560266718268e-05),
     'y': np.float64(9.127964585786685e-05),
     'z': np.float64(0.06000000238418579),
     'ux': np.float64(-0.00018198943225671794),
     'uy': np.float64(-0.0014729790270923558),
     'uz': np.float64(0.540184298489446),
     'time': np.float64(0.0010670959018170834),
     'weight': np.float64(1.0),
     'pdgcode': np.float64(2112.0)},
    'std': {'ekin': np.float64(0.0),
     'x': np.float64(0.29107582569122314),
     'y': np.float64(0.2913987338542938),
     'z': np.float64(1.8609769281141553e-09),
     'ux': np.float64(0.564337145136777),
     'uy': np.float64(0.5649386534107262),
     'uz': np.float64(0.26564446065932257),
     'time': np.float64(0.0009084215853363276),
     'weight': np.float64(0.0),
     'pdgcode': np.float64(0.0)}}, 20000: {'elapsed': 0,
    'mean': {'ekin': np.float64(2.5000000292152436e-08),
     'x': np.fl

In [45]:
original_data_metrics

{5: {'mean': {'ekin': np.float64(2.5000000292152436e-08),
   'x': np.float64(2.109775050485041e-05),
   'y': np.float64(-5.635877755594265e-07),
   'z': np.float64(0.06000000238418579),
   'ux': np.float64(4.754960143135579e-06),
   'uy': np.float64(-0.00022611875706264581),
   'uz': np.float64(0.5403950974286488),
   'time': np.float64(0.0010678209364414215),
   'weight': np.float64(1.0),
   'pdgcode': np.float64(2112.0)},
  'std': {'ekin': np.float64(0.0),
   'x': np.float64(0.2901604473590851),
   'y': np.float64(0.29015448689460754),
   'z': np.float64(3.3455975767537893e-09),
   'ux': np.float64(0.5644895487373589),
   'uy': np.float64(0.5644903035839511),
   'uz': np.float64(0.2658483907309259),
   'time': np.float64(0.0009099651942960918),
   'weight': np.float64(0.0),
   'pdgcode': np.float64(0.0)}},
 6: {'mean': {'ekin': np.float64(0.6281302615332215),
   'x': np.float64(-17.815603256225586),
   'y': np.float64(-0.2924065589904785),
   'z': np.float64(286.9150695800781),
   'u

### Visualize metrics

In [52]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

file_widget = widgets.Dropdown(options=list(gen_data_metrics.keys()), description='file:')
prop_widget = widgets.Dropdown(options=["mean", "std", "elapsed"], description='prop:')
dim_widget = widgets.Dropdown(options=["ekin", "x", "y", "z", "ux", "uy", "uz", "weight", "pdgcode"], description='dimension:')

plot_button = widgets.Button(description="Generate Plot", button_style='primary')

out = widgets.Output()

_is_running = False

def on_button_clicked(b):
    global _is_running
    if _is_running:
        return

    _is_running = True
    b.description = "Plotting..."
    b.disabled = True

    try:
        with out:
            clear_output(wait=True)

            file = file_widget.value
            prop = prop_widget.value
            dimension = dim_widget.value

            gen_metrics = gen_data_metrics[file]
            orig_metrics = original_data_metrics[file]

            bw_kernels = list(gen_metrics.keys())
            bws = sorted(list({bw_kernel[0] for bw_kernel in bw_kernels}))
            kernels = sorted(list({bw_kernel[1] for bw_kernel in bw_kernels}))
            # print(bws, kernels)
            
            n_bws, n_kernels = len(bws), len(kernels)
            # print(n_bws, n_kernels)
            
            fig, axs = plt.subplots(n_bws, n_kernels, figsize=(7*n_kernels, 5*n_bws), squeeze=False)
            for i_bw, bw in enumerate(bws):
                for i_kernel, kernel in enumerate(kernels):
                    ax = axs[i_bw, i_kernel]
                    bw_kernel = (bw, kernel)
                    metrics = gen_metrics[bw_kernel]
                    n_fits = sorted(list(metrics.keys()))
                    # print(n_fits)
                    ylabel = prop
                    if prop == "elapsed":
                        prop_values = [metrics[n_fit][prop] for n_fit in n_fits]
                        # print(prop_values)
                        ylabel += " (s)"
                        ax.set_yscale("symlog")
                    else:
                        prop_values = [metrics[n_fit][prop][dimension] for n_fit in n_fits]
                        ylabel += " "+dimension
                        orig_value = orig_metrics[prop][dimension]
                        ax.axhline(orig_value, color='gray', linestyle='--', label="orig. data")
                    ax.plot(n_fits, prop_values, marker='o', label="gen. data")
                    ax.set_title(bw+" "+kernel)
                    ax.set_xlabel("n_fit")
                    ax.set_ylabel(ylabel)
                    ax.set_xscale("log")
                    ax.legend()
                    ax.grid()

            fig.suptitle(example_paths[file])
            plt.tight_layout()
            plt.show()
            plt.close(fig)
            return
    finally:
        # Always release the lock and restore the button even if something errors
        _is_running = False
        b.description = "Generate Plot"
        b.disabled = False

# Link the button click to the function
plot_button._click_handlers.callbacks.clear()
plot_button.on_click(on_button_clicked)

# Display the UI block
display(widgets.VBox([file_widget, prop_widget, dim_widget, plot_button, out]))